# Short-Term Crypto Strategy Backtest

This notebook implements the hybrid trend–momentum–volatility strategy for 5–30 min charts:

- **Trend filter:** Price vs. 200 EMA  
- **Entry trigger:** Heikin-Ashi candle color + Parabolic SAR flip + RSI filter  
- **Stop-loss:** 2 × ATR(14)  
- **Exit:** PSAR flip trailing stop  


In [12]:
import yfinance as yf
import pandas as pd
import pandas_ta as ta

from backtesting import Backtest, Strategy

In [ ]:
# def import_data(stock, startDate):
#     df = yf.download(stock, start=startDate, interval='15m')
#     df.columns = df.columns.get_level_values(0)

#     # 200-period EMA on Close
#     df["EMA200"] = ta.ema(df["Close"], length=200)

#     # Heikin-Ashi candles
#     ha = ta.ha(df["Open"], df["High"], df["Low"], df["Close"])
#     # ha returns a DataFrame with columns: HA_open, HA_high, HA_low, HA_close
#     df = pd.concat([df, ha], axis=1)

#     # Parabolic SAR (long and short lines)
#     psar = ta.psar(df["High"], df["Low"], df["Close"])
#     # PSARl = long-side SAR; PSARs = short-side SAR
#     df["PSARl"] = psar["PSARl_0.02_0.2"]
#     df["PSARs"] = psar["PSARs_0.02_0.2"]

#     # 14-period RSI
#     df["RSI14"] = ta.rsi(df["Close"], length=14)

#     # 14-period ATR for volatility-based stops
#     df["ATR14"] = ta.atr(df["High"], df["Low"], df["Close"], length=14)

#     return df

In [20]:
import finnhub
import pandas as pd

def import_data(symbol, start_date):
    # Initialize client
    api_key = "d0n7th9r01qmjqml8pdgd0n7th9r01qmjqml8pe0"
    client = finnhub.Client(api_key=api_key)

    # 1. Fetch historical daily prices for AAPL
    res = client.stock_candles('AAPL', 'D')
    df_stocks = pd.DataFrame({
        't': pd.to_datetime(res['t'], unit='s'),
        'open': res['o'],
        'high': res['h'],
        'low':  res['l'],
        'close':res['c'],
        'volume': res['v']
    })
    print(df_stocks.tail())

    # 2. Fetch latest crypto price for BTC/USD
    btc_quote = client.crypto_price_quote('BINANCE', 'BTCUSDT')
    print(f"BTC/USD: bid={btc_quote['b']}, ask={btc_quote['a']}")
    return df_stocks

In [21]:
symbol     = "BTC-USD"
start_date = "2021-01-01"
interval   = "15m"

df = import_data(symbol, start_date)

# Drop any rows with missing data after download
df.dropna(inplace=True)

TypeError: Client.stock_candles() missing 2 required positional arguments: '_from' and 'to'

In [ ]:
class CryptoPSARStrategy(Strategy):
    """
    Hybrid trend + momentum + volatility strategy:
     - Trend filter: Price vs EMA200
     - Entry: Heikin Ashi candle flip + PSAR flip + RSI filter
     - Stop: 2 × ATR14
     - Exit: PSAR reversal trailing stop
    """
    def init(self):
        # Cache series for speed
        self.ema200 = self.data["EMA200"]
        self.ha_open = self.data["HA_open"]
        self.ha_close = self.data["HA_close"]
        self.psar_l = self.data["PSARl"]
        self.psar_s = self.data["PSARs"]
        self.rsi14  = self.data["RSI14"]
        self.atr14  = self.data["ATR14"]

    def next(self):
        price = self.data.Close[-1]

        # If no open position, check for entry
        if not self.position:
            # LONG ENTRY conditions
            cond_trend_up    = price > self.ema200[-1]
            cond_ha_bull     = self.ha_close[-1] > self.ha_open[-1]
            # PSAR flip: previous bar PSAR > price && current PSAR < price
            cond_psar_flip_l = (self.psar_l[-2] > self.data.Close[-2]) and (self.psar_l[-1] < price)
            cond_rsi_ok      = self.rsi14[-1] < 50

            if cond_trend_up and cond_ha_bull and cond_psar_flip_l and cond_rsi_ok:
                # Set stop-loss 2 × ATR below entry
                stop_price = price - 2 * self.atr14[-1]
                self.buy(sl=stop_price)

            # SHORT ENTRY conditions (mirror logic)
            cond_trend_dn    = price < self.ema200[-1]
            cond_ha_bear     = self.ha_close[-1] < self.ha_open[-1]
            cond_psar_flip_s = (self.psar_s[-2] < self.data.Close[-2]) and (self.psar_s[-1] > price)
            cond_rsi_ok_s    = self.rsi14[-1] > 50

            if cond_trend_dn and cond_ha_bear and cond_psar_flip_s and cond_rsi_ok_s:
                stop_price = price + 2 * self.atr14[-1]
                self.sell(sl=stop_price)

        # If in a position, exit on PSAR reversal
        else:
            if self.position.is_long:
                # PSAR dot flips above price → exit long
                if self.psar_l[-1] > price:
                    self.position.close()
            else:
                # PSAR dot flips below price → exit short
                if self.psar_s[-1] < price:
                    self.position.close()

In [ ]:
bt = Backtest(
    df,
    CryptoPSARStrategy,
    cash=10_000,
    commission=0.0005,   # ~0.05% per trade side
    trade_on_close=True, # execute orders at close price of signal bar
    exclusive=True       # only one position at a time
)

stats = bt.run()
print(stats)
bt.plot()  # opens an interactive equity curve & trade chart
